In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")


💻 Local Lab Server Environment Loaded.


## 1. Load Data

In [2]:
import json
import random
import copy
from sklearn.model_selection import train_test_split
import pandas as pd

TRAIN_PATH = f"{codebase_path}/data/train.jsonl"
TEST_PATH = f"{codebase_path}/data/test.jsonl"
DEV_PATH = f"{codebase_path}/data/addition.jsonl"
os.makedirs(DATA_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(TRAIN_PATH)
dev_data = load_jsonl(DEV_PATH)
test_data = load_jsonl(TEST_PATH)
print(f"Train samples: {len(train_data)}")
print(f"Dev samples: {len(dev_data)}")


Train samples: 13587
Dev samples: 4824


## 2. Option Expansion (Hard Negatives)

In [ ]:
# We will collect all tools from dev_data by domain
tools_by_domain = {}
for item in dev_data:
    domain = item.get('domain', 'Unknown')
    if domain not in tools_by_domain:
        tools_by_domain[domain] = []
    for tool in item['tools']:
        # Store uniquely by name
        if not any(t['name'] == tool['name'] for t in tools_by_domain[domain]):
            tools_by_domain[domain].append(tool)

# Fallback pool
all_tools = []
for d in tools_by_domain.values():
    all_tools.extend(d)

print(f"Total domains found: {len(tools_by_domain)}")
print(f"Total unique tools in dev: {len(all_tools)}")


In [ ]:
# Expand train_data to have 8 options
import string
letters = list(string.ascii_uppercase)[:8] # A-H

def expand_options(data, target_length=8):
    expanded = []
    for row in data:
        new_row = copy.deepcopy(row)
        current_options = list(new_row['options'].values())
        correct_letter = new_row['answer']
        correct_tool = new_row['options'][correct_letter]
        
        num_needed = target_length - len(current_options)
        if num_needed <= 0:
            expanded.append(new_row)
            continue
            
        current_names = set(t['name'] for t in current_options)
        candidates = [t for t in all_tools if t['name'] not in current_names]
        random.shuffle(candidates)
        
        added_tools = candidates[:num_needed]
        all_options = current_options + added_tools
        random.shuffle(all_options)
        
        new_options_dict = {}
        new_answer = None
        for i, opt in enumerate(all_options):
            letter = letters[i]
            new_options_dict[letter] = opt
            if opt['name'] == correct_tool['name']:
                new_answer = letter
                
        new_row['options'] = new_options_dict
        new_row['answer'] = new_answer
        expanded.append(new_row)
    return expanded

train_expanded = expand_options(train_data)
print("Finished expanding options to A-H.")


## 3. Structural-Only Stripping

In [ ]:
def strip_descriptions(obj):
    if isinstance(obj, dict):
        new_dict = {}
        for k, v in obj.items():
            if k == 'description':
                continue
            new_dict[k] = strip_descriptions(v)
        return new_dict
    elif isinstance(obj, list):
        return [strip_descriptions(item) for item in obj]
    else:
        return obj

def make_structural(data):
    structural = []
    for row in data:
        new_row = copy.deepcopy(row)
        new_row['options'] = strip_descriptions(new_row['options'])
        structural.append(new_row)
    return structural

train_structural = make_structural(train_expanded)
print("Finished stripping descriptions for Structural configuration.")


## 4. Stratified Validation Split & Caching

In [ ]:
# Approximate stratifying by checking the correct tool's name
tool_to_domain = {}
for item in dev_data:
    domain = item.get('domain', 'Unknown')
    for tool in item['tools']:
        tool_to_domain[tool['name']] = domain

def get_domain(row):
    correct_tool = row['options'][row['answer']]
    return tool_to_domain.get(correct_tool['name'], 'Unknown')

domains = [get_domain(row) for row in train_expanded]

train_idx, val_idx = train_test_split(
    range(len(train_expanded)), 
    test_size=0.15, 
    stratify=domains,
    random_state=42
)

# Split Full Info
train_full = [train_expanded[i] for i in train_idx]
val_full = [train_expanded[i] for i in val_idx]

# Split Structural
train_struct = [train_structural[i] for i in train_idx]
val_struct = [train_structural[i] for i in val_idx]

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

save_jsonl(train_full, f"{DATA_DIR}/train_full_info.jsonl")
save_jsonl(val_full, f"{DATA_DIR}/val_full_info.jsonl")
save_jsonl(train_struct, f"{DATA_DIR}/train_structural.jsonl")
save_jsonl(val_struct, f"{DATA_DIR}/val_structural.jsonl")

print("✅ Saved all dataset permutations to output/cache/")
